<a href="https://colab.research.google.com/github/felipefe/python-version/blob/stocks/knn_stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [204]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [205]:
# Parametros de entrada
data_inicio = "2022-01-01"

# ticker_a = "BBDC4.SA"
# ticker_b = "SANB11.SA"

ticker_a = "SUZB3.SA"
ticker_b = "KLBN4.SA"

# ticker_a = "VALE3.SA"
# ticker_b = "GGBR4.SA"

In [206]:
# 1. Coleta de dados semanais para o par ITUB4 e BBDC4
df_a = yf.download(ticker_a, start=data_inicio, interval="1wk", progress=False, auto_adjust=True)
df_b = yf.download(ticker_b, start=data_inicio, interval="1wk", progress=False, auto_adjust=True)

In [207]:
# Limpeza caso o yfinance retorne MultiIndex
if isinstance(df_a.columns, pd.MultiIndex):
  df_a.columns = df_a.columns.get_level_values(0)
if isinstance(df_b.columns, pd.MultiIndex):
  df_b.columns = df_b.columns.get_level_values(0)

In [208]:
# Alinhando as datas dos dois ativos em um único DataFrame
df = pd.DataFrame({"Close_A": df_a["Close"], "Close_B": df_b["Close"]}).dropna()

In [209]:
# 2. Engenharia de Features para Pairs Trading
# Ratio de preços entre os ativos
df["Ratio"] = df["Close_A"] / df["Close_B"]

# Média Móvel e Desvio Padrão do Ratio (Janela de 20 semanas)
window = 20
df["Ratio_Mean"] = df["Ratio"].rolling(window).mean()
df["Ratio_Std"] = df["Ratio"].rolling(window).std()

In [210]:
# Z-Score do Spread (nossa principal feature de desvio)
df["Z_Score"] = (df["Ratio"] - df["Ratio_Mean"]) / df["Ratio_Std"]

# Velocidade de variação do Z-Score na última semana (Z_Diff)
df["Z_Diff"] = df["Z_Score"].diff()

# Volatilidade conjunta do ratio
df["Ratio_Vol"] = df["Ratio"].rolling(5).std()

In [211]:
# 3. Alvo (Target): O Z-Score da PRÓXIMA semana
df["Target_Z_Next"] = df["Z_Score"].shift(-1)

# Remover valores nulos gerados pelas janelas e pelo shift
features = ["Z_Score", "Z_Diff", "Ratio_Vol"]
df_model = df.dropna(subset=features + ["Target_Z_Next"]).copy()

X = df_model[features]
y = df_model["Target_Z_Next"]

In [212]:
# 4. Divisão Cronológica (70% treino, 30% teste)
split_idx = int(len(df_model) * 0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [213]:
from sklearn.neighbors import KNeighborsRegressor

# 5. Pipeline KNN Regressor
# n_neighbors=10 considera um histórico maior de semanas parecidas
# weights='distance' dá mais peso às semanas com configuração geométrica mais idêntica
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    (
        "knn",
        KNeighborsRegressor(n_neighbors=10, weights="distance", p=2),
    ),
])

pipeline_knn.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('knn',
                 KNeighborsRegressor(n_neighbors=10, weights='distance'))])

In [214]:
from sklearn.metrics import mean_squared_error

# 6. Avaliação do modelo no conjunto de teste
preds = pipeline_knn.predict(X_test)
mse = mean_squared_error(y_test, preds)
print(f"Erro Quadrático Médio (MSE) no teste: {mse:.4f}")

Erro Quadrático Médio (MSE) no teste: 0.9270


In [215]:
# 7. Sinal Operacional para a Próxima Semana
ultima_instancia = df[features].dropna().iloc[[-1]]
z_atual = ultima_instancia["Z_Score"].values[0]
z_previsto = pipeline_knn.predict(ultima_instancia)[0]

In [216]:
print(f"\n--- Análise de Pairs Trading: {ticker_a} vs {ticker_b} ---")
print(f"Data de Referência: {ultima_instancia.index[0].strftime('%Y-%m-%d')}")
print(f"Z-Score Atual do Spread: {z_atual:.2f}")
print(f"Z-Score Previsto pelo KNN (Próxima Semana): {z_previsto:.2f}")


# Lógica de Decisão Operacional
limiar_entrada = 1.5
if z_atual > limiar_entrada and z_previsto < z_atual:
  print(
      f"\nSINAL DE VENDA DO SPREAD (Reversão para Baixo):"
      f"\n-> O spread está esticado acima de +{limiar_entrada} e o KNN prevê"
      f" fechamento para {z_previsto:.2f}."
      f"\n-> Ação Prática: Vender {ticker_a} e Comprar {ticker_b}."
  )
elif z_atual < -limiar_entrada and z_previsto > z_atual:
  print(
      f"\nSINAL DE COMPRA DO SPREAD (Reversão para Alta):"
      f"\n-> O spread está esticado abaixo de -{limiar_entrada} e o KNN prevê"
      f" fechamento para {z_previsto:.2f}."
      f"\n-> Ação Prática: Comprar {ticker_a} e Vender {ticker_b}."
  )
else:
  print(
      "\nSINAL: Neutro. O spread está dentro da zona de normalidade ou o"
      " modelo não aponta reversão forte."
  )


--- Análise de Pairs Trading: SUZB3.SA vs KLBN4.SA ---
Data de Referência: 2026-11-07
Z-Score Atual do Spread: 2.70
Z-Score Previsto pelo KNN (Próxima Semana): -0.39

SINAL DE VENDA DO SPREAD (Reversão para Baixo):
-> O spread está esticado acima de +1.5 e o KNN prevê fechamento para -0.39.
-> Ação Prática: Vender SUZB3.SA e Comprar KLBN4.SA.


Teste de Engle-Granger: No mercado financeiro, existe um erro clássico: achar que correlação é o mesmo que cointegração.
Correlação alta significa apenas que dois ativos sobem e descem na mesma direção ao mesmo tempo (ex: Petrobras e Vale podem subir juntas porque o Ibovespa subiu).
Cointegração vai muito além: significa que, embora os preços individuais de dois ativos possam vagar livremente por aí, a distância (o spread) entre eles é estacionária, ou seja, ela tem uma força elástica natural que sempre a puxa de volta para uma média histórica. Se você operar um par que é apenas correlacionado, o spread pode abrir e nunca mais voltar, quebrando sua conta.O Teste de Engle-Granger (baseado no teste estatístico ADF - Augmented Dickey-Fuller) é a ferramenta matemática que valida cientificamente se dois ativos formam um par cointegrado. Ele funciona em dois passos:Roda uma Regressão Linear entre o preço do Ativo A e o Ativo B para encontrar a proporção ideal de hedge (o coeficiente $\beta$).Aplica o teste ADF nos resíduos (o spread) dessa regressão para provar que a série temporal do spread não tem raiz unitária (ou seja, flutua ao redor de uma média fixa).

In [217]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import yfinance as yf

# 1. Baixar dados
df_a = yf.download(
    ticker_a, start=data_inicio, interval="1wk", progress=False, auto_adjust=True
)["Close"]
df_b = yf.download(
    ticker_b, start=data_inicio, interval="1wk", progress=False, auto_adjust=True
)["Close"]

# Garantir formato Series caso o yfinance retorne DataFrame MultiIndex
if isinstance(df_a, pd.DataFrame):
  df_a = df_a.iloc[:, 0]
if isinstance(df_b, pd.DataFrame):
  df_b = df_b.iloc[:, 0]

# Montando o DataFrame dinamicamente com os nomes dos tickers
df = pd.DataFrame({ticker_a: df_a, ticker_b: df_b}).dropna()

# Passo 1: Regressão Linear OLS dinamicamente
X = sm.add_constant(df[ticker_b])
model = sm.OLS(df[ticker_a], X).fit()
beta = model.params[ticker_b]
alpha = model.params["const"]

# Calcula o Spread (Resíduos) de forma dinâmica
spread = df[ticker_a] - (beta * df[ticker_b] + alpha)

# Passo 2: Teste ADF (Augmented Dickey-Fuller) no Spread
adf_result = adfuller(spread)

print(f"--- Análise de Cointegração: {ticker_a} vs {ticker_b} ---")
print(f"Hedge Ratio (Beta): {beta:.4f}")
print(f"Estatística ADF: {adf_result[0]:.4f}")
print(f"p-valor do Teste ADF: {adf_result[1]:.4f}")

if adf_result[1] < 0.05:
  print(
      f"-> O p-valor é menor que 5%: {ticker_a} e {ticker_b} SÃO cointegrados"
      " estatisticamente. Par apto para Pairs Trading!"
  )
else:
  print(
      f"-> O p-valor é maior que 5%: Não há evidência estatística de"
      f" cointegração entre {ticker_a} e {ticker_b} neste período."
  )

--- Análise de Cointegração: SUZB3.SA vs KLBN4.SA ---
Hedge Ratio (Beta): 10.4918
Estatística ADF: -3.3152
p-valor do Teste ADF: 0.0142
-> O p-valor é menor que 5%: SUZB3.SA e KLBN4.SA SÃO cointegrados estatisticamente. Par apto para Pairs Trading!


/tmp/ipykernel_4580/3624554386.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_result = adfuller(spread)
